# Experiment 12: Random Forest by flat-type segment

Compare a global Random Forest against one Random Forest per sufficiently large flat type. Each segment model is fitted only on that segment's training rows; sparse or unseen flat types use the global prediction.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'; sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

TARGET, GROUP_COLUMNS, LAG_MONTHS = 'MONTHLY_RENT', ['TOWN', 'FLAT_TYPE'], 24
MIN_SEGMENT_TRAIN_ROWS = int(os.environ.get('RF_MIN_SEGMENT_ROWS', '2000'))
RF_TREES = int(os.environ.get('RF_TREES', '300'))
compact_features = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
MODEL_FEATURES = compact_features + ['monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name', 'nearest_tourist_attraction_name', 'nearest_monument_name']

_, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy(); raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
periods = np.sort(raw_train.period.unique()); validation_cutoff = periods[int(len(periods) * .80)]
raw_fit, raw_valid = raw_train.loc[raw_train.period < validation_cutoff].copy(), raw_train.loc[raw_train.period >= validation_cutoff].copy()
print(f'Cutoff={validation_cutoff}; fit={len(raw_fit):,}; validation={len(raw_valid):,}; trees={RF_TREES}; segment minimum={MIN_SEGMENT_TRAIN_ROWS:,}')


Cutoff=2024-05; fit=117,988; validation=32,012; trees=300; segment minimum=2,000


In [2]:
class TargetFeatureTransformer:
    def fit(self, history):
        self.global_mean_ = history[TARGET].mean(); self.group_mean_ = history.groupby(GROUP_COLUMNS)[TARGET].mean(); self.monthly_mean_ = history.groupby('period')[TARGET].mean()
        self.group_period_ = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].mean().rename(columns={'period':'source_period', TARGET:'town_flat_type_target_lag'}); return self
    def transform(self, history):
        result = history[GROUP_COLUMNS + ['period']].copy(); result['monthly_target_lag'] = (result.period-LAG_MONTHS).map(self.monthly_mean_).fillna(self.global_mean_)
        result['source_period'] = result.period-LAG_MONTHS; result = result.merge(self.group_period_, on=GROUP_COLUMNS+['source_period'], how='left').drop(columns='source_period')
        means = pd.Series(result.set_index(GROUP_COLUMNS).index.map(self.group_mean_), index=result.index); result['town_flat_type_target_mean'] = means.fillna(self.global_mean_)
        return result[['monthly_target_lag','town_flat_type_target_mean']]
    def fit_transform_train(self, history):
        self.fit(history); grouped = history.groupby(GROUP_COLUMNS+['period'], as_index=False)[TARGET].agg(['sum','count']).reset_index().sort_values(GROUP_COLUMNS+['period'])
        grouped['prior_sum'] = grouped.groupby(GROUP_COLUMNS)['sum'].cumsum()-grouped['sum']; grouped['prior_count'] = grouped.groupby(GROUP_COLUMNS)['count'].cumsum()-grouped['count']
        grouped['town_flat_type_target_mean'] = grouped.prior_sum/grouped.prior_count.replace(0,np.nan)
        global_monthly = history.groupby('period')[TARGET].agg(['sum','count']).sort_index(); global_prior = (global_monthly['sum'].cumsum()-global_monthly['sum'])/(global_monthly['count'].cumsum()-global_monthly['count']).replace(0,np.nan)
        result = history[GROUP_COLUMNS+['period']].merge(grouped[GROUP_COLUMNS+['period','town_flat_type_target_mean']], on=GROUP_COLUMNS+['period'], how='left'); result['town_flat_type_target_mean'] = result.town_flat_type_target_mean.fillna(result.period.map(global_prior))
        result['monthly_target_lag'] = (result.period-LAG_MONTHS).map(self.monthly_mean_).fillna(result.period.map(global_prior)); return result[['monthly_target_lag','town_flat_type_target_mean']]

def preprocess_partition(preprocessor, frame, fit=False):
    values = frame.drop(columns=['period', TARGET], errors='ignore'); return preprocessor.fit_transform(values) if fit else preprocessor.transform(values)
def labelled_history(frame, base):
    history = base[GROUP_COLUMNS].reset_index(drop=True).copy(); history['period'] = frame.period.to_numpy(); history[TARGET] = frame[TARGET].to_numpy(); return history
def nearest_mrt_name(base):
    stations = datasets['mrt'][['NAME','LATITUDE','LONGITUDE']].dropna(); _, index = cKDTree(stations[['LATITUDE','LONGITUDE']].to_numpy()*[111,110.96]).query(base[['LATITUDE','LONGITUDE']].to_numpy()*[111,110.96]); return pd.Series(stations.NAME.astype(object).to_numpy()[index], name='nearest_mrt_name')
def nearest_geojson_name(base, filename, name_field, feature_name):
    with (project_root/'data'/'possible_additional_features'/filename).open() as handle: data=json.load(handle)
    points=pd.DataFrame([{'longitude':f['geometry']['coordinates'][0],'latitude':f['geometry']['coordinates'][1],feature_name:f['properties'][name_field]} for f in data['features'] if f.get('geometry',{}).get('type')=='Point']).dropna()
    _, index=cKDTree(points[['longitude','latitude']].to_numpy()*[110.96,111]).query(base[['LONGITUDE','LATITUDE']].to_numpy()*[110.96,111]); return pd.Series(points[feature_name].astype(object).to_numpy()[index], name=feature_name)

preprocessor = RentalPreprocessor(datasets); fit_base = preprocess_partition(preprocessor, raw_fit, fit=True); valid_base = preprocess_partition(preprocessor, raw_valid)
transformer = TargetFeatureTransformer(); fit_target = transformer.fit_transform_train(labelled_history(raw_fit, fit_base)); valid_target = transformer.transform(labelled_history(raw_valid, valid_base))
def model_frame(base, target_features):
    return pd.concat([base[compact_features].reset_index(drop=True), target_features.reset_index(drop=True), nearest_mrt_name(base).reset_index(drop=True), nearest_geojson_name(base,'TouristAttractions.geojson','PAGETITLE','nearest_tourist_attraction_name').reset_index(drop=True), nearest_geojson_name(base,'MonumentsGEOJSON.geojson','NAME','nearest_monument_name').reset_index(drop=True)],axis=1)[MODEL_FEATURES]
fit_x, valid_x = model_frame(fit_base,fit_target), model_frame(valid_base,valid_target); fit_y, valid_y = raw_fit[TARGET].astype(float).reset_index(drop=True), raw_valid[TARGET].astype(float).reset_index(drop=True)
assert len(fit_x)==len(fit_y) and len(valid_x)==len(valid_y)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [3]:
categorical_columns = fit_x.select_dtypes(include=['object','category','string']).columns.tolist(); numeric_columns = [column for column in fit_x.columns if column not in categorical_columns]
def make_random_forest():
    encoder = ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median'))]),numeric_columns),('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical_columns)])
    forest = RandomForestRegressor(n_estimators=RF_TREES, min_samples_leaf=3, max_features=0.7, n_jobs=-1, random_state=42)
    return Pipeline([('encode',encoder),('forest',forest)])
def metric_row(name, actual, prediction): return {'model':name,'RMSE':mean_squared_error(actual,prediction)**.5,'MAE':mean_absolute_error(actual,prediction),'R2':r2_score(actual,prediction)}

global_forest = make_random_forest().fit(fit_x,fit_y); global_prediction = global_forest.predict(valid_x)
segmented_prediction = global_prediction.copy(); route = pd.Series('global_fallback',index=valid_x.index,dtype=object)
coverage = fit_x.groupby('FLAT_TYPE').size(); eligible_flat_types = coverage.index[coverage.ge(MIN_SEGMENT_TRAIN_ROWS)].tolist(); segment_forests={}
for flat_type in eligible_flat_types:
    train_mask, valid_mask = fit_x.FLAT_TYPE.eq(flat_type), valid_x.FLAT_TYPE.eq(flat_type)
    if not valid_mask.any(): continue
    forest = make_random_forest().fit(fit_x.loc[train_mask],fit_y.loc[train_mask])
    segmented_prediction[valid_mask] = forest.predict(valid_x.loc[valid_mask]); route.loc[valid_mask] = f'flat_type_{flat_type}'; segment_forests[flat_type]=forest

comparison = pd.DataFrame([metric_row('Global Random Forest',valid_y,global_prediction),metric_row('Flat-type routed Random Forest',valid_y,segmented_prediction)])
display(comparison.sort_values('RMSE')); display(pd.DataFrame({'fit_rows':coverage,'eligible':coverage.ge(MIN_SEGMENT_TRAIN_ROWS)})); display(route.value_counts().rename_axis('route').reset_index(name='validation_rows'))
per_segment = pd.DataFrame([metric_row(flat_type,valid_y.loc[valid_x.FLAT_TYPE.eq(flat_type)],segmented_prediction[valid_x.FLAT_TYPE.eq(flat_type)]) for flat_type in sorted(valid_x.FLAT_TYPE.dropna().unique())]); display(per_segment.sort_values('RMSE'))


,model,RMSE,MAE,R2
0,Global Random Forest,484.923452,354.787439,0.455214
1,Flat-type routed Random Forest,486.073078,357.986641,0.452628


,fit_rows,eligible
FLAT_TYPE,,
1 room,32,False
2 room,2242,True
3 room,38440,True
4 room,42401,True
5 room,28253,True
executive,6620,True


,route,validation_rows
0,flat_type_4 room,11678
1,flat_type_3 room,10584
2,flat_type_5 room,7270
3,flat_type_executive,1614
4,flat_type_2 room,856
5,global_fallback,10


,model,RMSE,MAE,R2
0,1 room,358.439759,321.705368,-0.102825
1,2 room,449.313487,324.097961,0.019284
2,3 room,453.582611,342.601518,0.132662
3,4 room,491.736418,365.164708,0.274984
4,5 room,497.966936,354.836290,0.304229
5,executive,602.152627,439.328186,0.077496
